<a href="https://colab.research.google.com/github/LordRelentless/NGFTSimulations/blob/main/NGFT_Gravitational_Waves.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Simulation 4.4: NGFT Gravitational Wave Dissipation
# Language: Julia

using Plots
using Printf
gr()

println("--- NGFT Gravitational Wave Simulation ---")

# --- 1. Simulation and NGFT Parameters ---
N_FRAMES = 500
DT = 0.1
LATTICE_SIZE = 1000

# Wave Parameters
const WAVE_AMPLITUDE_INITIAL = 1.0
const WAVE_K = 0.1  # Wavenumber (controls frequency)
const WAVE_SPEED = 5.0

# Decay Parameters
# Represents the 1/r geometric spreading effect, affects both waves
const GEOMETRIC_DECAY = 0.005
# Represents the unique NGFT dissipative effect from ZPT interaction
const NGFT_DISSIPATION_COEFFICIENT = 0.003

# --- 2. PHASE 1: Pre-calculate Wave Envelopes ---
println("Phase 1: Pre-calculating wave amplitude envelopes...")
x_coords = 1:LATTICE_SIZE

# GR amplitude only decreases due to geometric spreading
amplitude_gr = WAVE_AMPLITUDE_INITIAL ./ (1.0 .+ GEOMETRIC_DECAY .* x_coords)

# NGFT amplitude has geometric decay AND the new dissipative effect
dissipation_factor = exp.(-NGFT_DISSIPATION_COEFFICIENT .* x_coords)
amplitude_ngft = amplitude_gr .* dissipation_factor

# --- 3. PHASE 2: Render the Animation ---
println("Phase 2: Rendering animation...")

anim = @animate for i in 1:N_FRAMES
    # A. Setup plot layout
    p_layout = @layout [a; b]
    p = plot(layout=p_layout, size=(1200, 800), background_color=:black,
             bottom_margin=10Plots.mm, left_margin=10Plots.mm)

    # B. Top Pane: Propagating Waves
    plot!(p[1], x_coords, zeros(LATTICE_SIZE), lw=1, color=:white, alpha=0.3, label="") # Center line
    plot!(p[1], xlims=(0, LATTICE_SIZE), ylims=(-1.2, 1.2),
          title="Gravitational Wave Propagation (Time: $(round(i*DT, digits=1)))",
          xlabel="Distance", ylabel="Strain Amplitude")

    # Calculate the wave's phase at this moment in time
    time = i * DT
    phase_offset = WAVE_SPEED * time

    # Calculate the y-positions for each wave
    y_gr = amplitude_gr .* sin.(WAVE_K .* x_coords .- phase_offset)
    y_ngft = amplitude_ngft .* sin.(WAVE_K .* x_coords .- phase_offset)

    plot!(p[1], x_coords, y_gr, color=:red, ls=:dash, lw=2, label="GR Wave (Geometric Decay Only)")
    plot!(p[1], x_coords, y_ngft, color=:cyan, lw=2, label="NGFT Wave (with ZPT Dissipation)")

    # C. Bottom Pane: Amplitude Envelopes (The Scientific Result)
    plot!(p[2], xlims=(0, LATTICE_SIZE), ylims=(0, WAVE_AMPLITUDE_INITIAL * 1.1),
          title="Amplitude Decay vs. Distance",
          xlabel="Distance from Source", ylabel="Peak Amplitude")

    plot!(p[2], x_coords, amplitude_gr, color=:red, ls=:dash, lw=3, label="GR Amplitude Envelope (1/r)")
    plot!(p[2], x_coords, amplitude_ngft, color=:cyan, lw=3, label="NGFT Amplitude Envelope (1/r * e⁻ˡˣ)")

    annotate!(p[2], LATTICE_SIZE/2, 0.8, text("NGFT predicts additional energy loss,\ncausing faster amplitude decay.",
                                             color=:white, pointsize=10))
    global last_plot = p # Store the last plot object
end

# --- 4. Save Final Outputs ---
gif(anim, "ngft_gravitational_wave.gif", fps=30)
println("Animation saved as ngft_gravitational_wave.gif")
savefig(last_plot, "ngft_gravitational_wave_final.png") # Save the stored plot object
println("Final plot saved as ngft_gravitational_wave_final.png")
println("Simulation successfully finished.")

--- NGFT Gravitational Wave Simulation ---
Phase 1: Pre-calculating wave amplitude envelopes...
Phase 2: Rendering animation...
Animation saved as ngft_gravitational_wave.gif


[ Info: Saved animation to /content/ngft_gravitational_wave.gif


Final plot saved as ngft_gravitational_wave_final.png
Simulation successfully finished.
